# Functions, Iterators, and Generators

In [49]:
import bisect
import csv
import itertools
import math
from collections import namedtuple
from collections.abc import Callable, Generator, Iterable, Iterator, Mapping
from typing import Any, NamedTuple, cast

import ipytest
import pytest

ipytest.autoconfig()

* The core of functional programming is the use of *pure functions* to map values from the input domain to the output range.
    * A pure function has *no side effects*.
* Avoiding side effects can lead to reducing any dependence on *variable assignment* to maintain the state of our computations.
* We can't purge the assignment statement from the Python language, but we can reduce our dependence on stateful objects.
    * This means choosing built-in functions and data structures that don't require stateful operations.

## Writing pure functions

* A function with no **side effects** fits the pure mathematical abstraction of a function: there are no *global* changes to variables.
* To be pure, a function should also avoid changing the *state mutable objects*.
* There are many internal Python objects that are stateful, e.g., instances of the `file` class and other file-like objects.
* We can't easily eliminate all stateful Python objects.
    * We must strike a balance between managing state while still exploiting the strengths of functional design.
* We should always use the `with` statement to encapsulate stateful file objects into a well-defined scope.
* We should always avoid global file objects, global database connections, and the associated stateful object issues.

## Functions as first-class objects

* Python functions are first-class objects.
    * Function objects have a number of attributes; we can:
        * assign functions to variables,
        * pass functions as arguments, and
        * return functions as values.
    * We can use these techniques to write higher-order functions.
* A *callable object* helps us to create functions.
    * We can consider the callable class definition as a higher-order function.
    * We should avoid setting stateful class variables in the `__init__()` method.
    * One common application is to use an `__init__()` method to create objects that fit the Strategy design pattern.
* A class following the *Strategy design pattern* depends on other objects to provide an algorithm or parts of an algorithm.
    * This allows us to inject algorithmic details at runtime.

In [50]:
%%ipytest


class Mersenne1:
    def __init__(self, algorithm: Callable[[int], int]) -> None:
        # Use __init__() to save a reference to another function, algorithm.
        self.pow2 = algorithm

    def __call__(self, arg: int) -> int:
        return self.pow2(arg) - 1


# Candidate objects with identical function signatures to plug into the Mersenne1 class.


def shifty(b: int) -> int:
    # Uses a left shift of the bits.
    return 1 << b


def multy(b: int) -> int:
    # Uses a naive recursion multiplication.
    if b == 0:
        return 1
    return 2 * multy(b - 1)


def faster(b: int) -> int:
    # Uses a divide and conquer strategy that will perform log2(b) multiplications
    # instead of b multiplications.
    if b == 0:
        return 1
    if b % 2 == 1:
        return 2 * faster(b - 1)
    t = faster(b // 2)
    return t * t


@pytest.mark.parametrize("strategy", [shifty, multy, faster])
@pytest.mark.parametrize(
    ("n", "result"),
    [(2, 3), (3, 7), (5, 31)],
)
def test_mersenne_strategy(
    strategy: Callable[[int], int],
    n: int,
    result: int,
) -> None:
    m1 = Mersenne1(strategy)
    assert m1(n) == result


.........                                                                                    [100%]


9 passed in 0.04s


## Using Strings

* Since Python strings are immutable, they're an excellent example of functional programming objects.
* A Python `str` object has a number of methods, all of which produce a new string as the result.
    * These methods are pure functions with no side effects.

## Using tuples and named tuples

* Since Python tuples are immutable objects, they're another excellent example of objects suitable for functional programming.
* The `namedtuple` class adds an essential feature to a tuple: a name that we can use instead of an index.
    * This allows us to write pure functions based on stateless objects, yet keep data bound into tidy object-like packages.
* We'll almost always use tuples (and named tuples) in the context of a collection of values.
* The decision to use a `tuple` or `namedtuple` object is entirely a matter of convenience.
  * E.g., consider working with a sequence of color values as a three tuple of the form `(number, number, number)`
    * It's not clear that these are in red, green, blue order.
    * We have a number of approaches for making the tuple structure explicit.

In [51]:
%%ipytest


def test_type_alias_tuple() -> None:
    # RGB is a type alias for a tuple of exactly 3 int elements.
    RGB = tuple[int, int, int]

    # We can clarify the triple structure by creating functions to pick a three-tuple
    # apart.
    red: Callable[[RGB], int] = lambda color: color[0]
    green: Callable[[RGB], int] = lambda color: color[1]
    blue: Callable[[RGB], int] = lambda color: color[2]

    color: RGB = (255, 128, 0)

    assert red(color) == 255
    assert green(color) == 128
    assert blue(color) == 0


def test_collections_namedtuple() -> None:
    # Using the older-style namedtuple class from the collections module
    Color = namedtuple("Color", ("red", "green", "blue", "name"))

    color = Color(255, 128, 0, "orange")

    assert color.red == 255
    assert color.green == 128
    assert color.blue == 0


def test_typing_namedtuple() -> None:
    # Using the NamedTuple class from the typing module
    class Color(NamedTuple):
        red: int
        green: int
        blue: int
        name: str

    color = Color(255, 128, 0, "orange")

    assert color.red == 255
    assert color.green == 128
    assert color.blue == 0

...                                                                                          [100%]
3 passed in 0.01s


## Using generator expressions

* A generator expression is a way to produce values lazily, one at a time, rather than creating an entire collection in memory upfront.
* List comprehensions (sometimes called a list display) create collections eagerly: `squares = [x**2 for x in range(10)]`
    * This evaluates the expression and creates a list containing all ten squares immediately.
* Generator expressions produce values lazily: `squares = (x**2 for x in range(10))`
    * This creates a generator that calculates each square as it is requested.
* Both are iterable, but they behave differently.
* Generators have limitations:
    * They generally cannot provide the length of the complete sequence using `len()`, because they do not store a collection of all their values.
    * They are normally exhausted after one complete iteration. To generate the values again, you need to create a new generator.
* Lazy evaluation can improve performance.
    * Generators can reduce memory consumption, particularly when processing large datasets or when only some of the available values are needed.
    * They do not necessarily make every computation faster, however.
* Generator expressions allow values to flow through the pipeline one at a time.
    * They enable lazy, memory-efficient data processing and allow functional-style transformation pipelines to operate without creating intermediate collections.
* In the following example generator function, we use one important **`for` loop** in addition to recursion.
    * Explicit loop allows us to easily handle numbers that have as many as 1,000 factors.
    * Since the for variable, `i`, is not used outside the indented body of the loop, the stateful nature of the `i` variable won't lead to confusion if we make any changes to the body of the loop.
* This example also shows how to do **tail-call optimization** manually.
    * The recursive calls that count from 3 to $\sqrt{n}$ have been replaced with a loop.
    * The `for` loop saves us from a deeply recursive call stack.

In [52]:
%%ipytest


def prime_factors_loop(x: int) -> Iterator[int]:
    """Locate the prime factors of a number."""
    # If the number, x, is even, yield 2 and then recursively yield all prime factors
    # of x/2.
    if x % 2 == 0:
        yield 2
        if x // 2 > 1:
            yield from prime_factors_loop(x // 2)
        return

    # For odd numbers, step through odd values greater than or equal to 3 to locate a
    # candidate factor of the number. When we locate a factor, i, yield that factor,
    # and then recursively yield all prime factors of x÷i.
    for i in range(3, int(math.sqrt(x) + 0.5) + 1, 2):
        if x % i == 0:
            yield i
            if x // i > 1:
                yield from prime_factors_loop(x // i)
            return

    # In the event that we can't locate a factor, x, must be prime, so we can yield it.
    yield x


def test_prime_factors_loop() -> None:
    num = 30

    prime_factors_values = prime_factors_loop(num)
    assert (a := next(prime_factors_values)) == 2
    assert (b := next(prime_factors_values)) == 3
    assert (c := next(prime_factors_values)) == 5
    assert a * b * c == num

    with pytest.raises(StopIteration):
        next(prime_factors_values)

.                                                                                            [100%]
1 passed in 0.01s


* The following example is a more purely recursive version.
    * While this is simpler than the `for` statement version, it can't handle numbers with over 1,000 factors because of Python's stack limitation.

In [53]:
%%ipytest


def prime_factors_recursive(x: int) -> Iterator[int]:

    def factor_n(x: int, n: int) -> Iterator[int]:
        """Test factors, n, in the range 3 <= n <= √x."""
        # If the candidate factor, n, is outside the range, then x is prime.
        if n * n > x:
            yield x
            return

        # Otherwise, see whether n is a factor of x.
        if x % n == 0:
            # If so, yield n and all factors of x/n.
            yield n
            if x // n > 1:
                yield from factor_n(x // n, n)
        else:
            # If not, evaluate the function recursively using n+2.
            yield from factor_n(x, n + 2)

    # Handle some edge cases.
    if x % 2 == 0:
        # For even numbers, we'll yield 2 and then evaluate prime_factors_recursive()
        # recursively for x÷2.
        yield 2
        if x // 2 > 1:
            yield from prime_factors_recursive(x // 2)
        return

    # All other prime factors must be odd numbers >= 3. Evaluate the factors_n()
    # function starting with 3 to test these other candidate prime factors.
    yield from factor_n(x, 3)


def test_prime_factors_recursive() -> None:
    num = 30
    result = list(prime_factors_recursive(num))
    assert result == [2, 3, 5]
    assert math.prod(result) == num


.                                                                                            [100%]
1 passed in 0.01s


### Supplementary notes on `yield from`

* **Delegating to another generator:**
    * `yield from` allows a generator to delegate its iteration to another iterable or generator.
    * Rather than writing a loop to yield each item individually, you can let Python handle that for you.

In [54]:
%%ipytest


def test_yield_from_delegation() -> None:
    squares = lambda n: (x * x for x in range(1, n + 1))

    def all_numbers_for_loop() -> Iterator[int]:
        yield 0

        # A loop to yield each item individually
        # Note: `squares(3)` returns a generator.
        for num in squares(3):
            yield num

        yield 10

    def all_numbers_yield_from() -> Iterator[int]:
        yield 0

        # Replaced loop with `yield from`; delegate part of the work to squares(3)
        yield from squares(3)

        yield 10

    expected = [0, 1, 4, 9, 10]
    assert list(all_numbers_for_loop()) == expected
    assert list(all_numbers_yield_from()) == expected


.                                                                                            [100%]
1 passed in 0.01s


* **`yield from` can return a value:**
    * A generator can return a final value using `return`.
    * When another generator delegates to it using `yield from`, that final value becomes the result of the `yield from` expression.

In [55]:
%%ipytest


def test_yield_from_return() -> None:
    def subgenerator() -> Generator[int, None, str]:
        yield 1
        yield 2
        return "finished"

    def main_generator() -> Iterator[int | str]:
        result = yield from subgenerator()
        yield result

    assert list(main_generator()) == [1, 2, "finished"]

.                                                                                            [100%]
1 passed in 0.01s


* **3 good uses of `yield from`:**
    1. *Generator composition:* You can construct a larger generator from smaller generators, each responsible for one part of a computation, for modularity and separation of concerns.
    2. *Lazy pipelines:* Generators can be combined without building large intermediate lists, which is useful when processing streams of data.
    3. *Delegation of behaviour:* The outer generator can delegate iteration to an inner generator without reimplementing its logic.

## Exploring the limitations of generators

* Generators are lazy, and doesn't have a proper value until we consume the results yielded by the generator, although in itself isn't a limitation.
* Because generators are lazy, their sizes can't be known until after all of the values are consumed.
* Generator functions can only be used once; they have a stateful life.
    * One way to overcome the once-only limitation is to use the `itertools.tee()` method.
    * `tee()` return `n` independent iterators from a single iterable.
        * Important memory usage implication: Any item pulled from the source iterable remains stored in memory until the *slowest* tee'd iterator finally consumes it.

In [56]:
%%ipytest


def test_tee() -> None:
    cubes = lambda n: (x * x * x for x in range(1, n + 1))
    five_cubes = cubes(5)

    cubes_a, cubes_b = itertools.tee(five_cubes, 2)
    # At this point, five_cubes is not yet consumed.

    # Start consuming the first tee'd iterator.
    assert len(list(cubes_a)) == 5
    assert len(list(cubes_a)) == 0

    # Original iterator is also consumed when any of the tee'd iterators is consumed.
    assert len(list(five_cubes)) == 0

    # The second tee'd iterator is still consumable.
    assert len(list(cubes_b)) == 5
    assert len(list(cubes_b)) == 0

.                                                                                            [100%]


1 passed in 0.01s


## Combining generator expressions

* The essence of functional programming comes from the ways we can easily combine generator expressions and generator functions to create very sophisticated composite processing sequences.
* Consider an example of processing a list of product prices:
    * $f(x)$: convert prices to include VAT, then
    * $g(x)$: apply a discount to the VAT-inclusive prices.
* The composite operation is $g(f(x))$: add VAT first, then apply the discount.
* This demonstrates:
    * function composition (applying one function after another),
    * reuse (using the intermediate result independently),
    * and lazy evaluation (calculating values only when requested).


In [57]:
%%ipytest


def test_composite_operation() -> None:

    def add_vat(price: float) -> float:
        """Add 20% VAT to a price."""
        return price * 1.20

    def apply_discount(price: float) -> float:
        """Apply a 10% discount."""
        return price * 0.90

    prices = [10.00, 25.00, 50.00, 100.00]

    f_x = (add_vat(x) for x in prices)
    # f_x can be reused independently of the composite generator, g_f_x.

    g_f_x = (apply_discount(y) for y in f_x)
    # When you request the first value from g_f_x, it requests a value from f_x, which
    # processes the first original price. The second generator then applies the
    # discount to that result.

    assert list(g_f_x) == [10.8, 27, 54, 108]

.                                                                                            [100%]
1 passed in 0.01s


## Using lists, dicts, and sets

* **Lists, tuples and other collections materialise data**
    * Lists and tuples store their elements, whereas iterators and generators produce values on demand.
    * List comprehensions and calls such as `list()` and `tuple()` consume an iterable and collect its values into a collection.
    * Simply placing an iterable inside a collection literal does not necessarily consume it.
        * E.g., `[range(10)]` creates a list containing one range object, whereas `[x for x in range(10)]` creates a list of ten integers.
* **Materialisation makes data reusable**
    * Iterators are consumed as their values are retrieved and generally cannot be reset for another pass.
    * Converting an iterator into a tuple or list allows its data to be traversed repeatedly.
    * In the [Anscombe's quartet](https://en.wikipedia.org/wiki/Anscombe%27s_quartet) example, the data is materialised as a tuple so that all four data series can be extracted from the same rows.
* **Use lazy processing for transformations, but materialise when necessary**
    * Build data-processing pipelines from small functions that perform individual transformations, such as filtering header rows, extracting columns and converting strings to numbers.
    * Use generator functions and expressions to process values lazily, without creating unnecessary intermediate collections.
    * Materialise the data when it needs to be stored or reused, rather than repeatedly consuming a one-shot iterator.
* **Materialisation has a cost**
    * Creating lists and tuples requires memory to store their elements and takes time.
    * Generators can reduce memory usage by computing values only when required, potentially improving performance.
    * Prefer lazy generators where practical, but materialise collections when necessary.

**Key insight:** The section advocates a *lazy by default, materialise when necessary* approach. Combine small, composable transformations to process data efficiently, but recognise that reusable data structures and carefully controlled state are sometimes essential.

In [58]:
%%ipytest

anscombe = """Anscombe's quartet
I	II	III	IV
x	y	x	y	x	y	x	y
10.0	8.04	10.0	9.14	10.0	7.46	8.0	6.58
8.0	6.95	8.0	8.14	8.0	6.77	8.0	5.76
13.0	7.58	13.0	8.74	13.0	12.74	8.0	7.71"""


def row_iter(source: str) -> Iterator[list[str]]:
    """Create an iterator over the rows."""
    return csv.reader(source.splitlines(), delimiter="\t")


def head_split_fixed(row_iter: Iterator[list[str]]) -> Iterator[list[str]]:
    """Filter first 3 non-data rows."""
    # Expected headers: (expected_length, expected_values)
    expected_headers = (
        (1, ["Anscombe's quartet"]),
        (4, ["I", "II", "III", "IV"]),
        (8, ["x", "y", "x", "y", "x", "y", "x", "y"]),
    )

    for expected_len, expected_vals in expected_headers:
        try:
            row = next(row_iter)
        except StopIteration:
            msg = "Data stream ended prematurely while validating headers."
            raise ValueError(msg) from None

        if len(row) != expected_len or row != expected_vals:
            msg = f"Invalid header format. Expected {expected_vals}, but got {row}."
            raise ValueError(msg)

    return row_iter


# Would be better to use NamedTuple here; covered in a later chapter.
Pair = tuple[str, str]


def series(pair_num: int, row_iter: Iterable[list[str]]) -> Iterator[Pair]:
    """Select pairs of columns.

    Pick two adjacent columns based on a number between 0 and 3.
    """
    for row in row_iter:
        index_offset = pair_num * 2
        yield cast("Pair", tuple(row[index_offset : index_offset + 2]))


# ---


def test_anscombe_stats_summary() -> None:
    # Data cleansing using a composite function.

    # Data is materialised as a tuple so that all 4 data series (I-IV) can be extracted
    # from the same rows.
    data = tuple(head_split_fixed(row_iter(anscombe)))

    assert data == (
        ["10.0", "8.04", "10.0", "9.14", "10.0", "7.46", "8.0", "6.58"],
        ["8.0", "6.95", "8.0", "8.14", "8.0", "6.77", "8.0", "5.76"],
        ["13.0", "7.58", "13.0", "8.74", "13.0", "12.74", "8.0", "7.71"],
    )

    # Further processing to pick x,y pairs.

    # Apply an overall tuple() function to materialise the resulting tuple-of-tuple
    # sequences so that we can do further processing on each one.
    sample_1 = tuple(series(0, data))
    sample_2 = tuple(series(1, data))
    sample_3 = tuple(series(2, data))
    sample_4 = tuple(series(3, data))

    assert sample_1 == (("10.0", "8.04"), ("8.0", "6.95"), ("13.0", "7.58"))
    assert sample_2 == (("10.0", "9.14"), ("8.0", "8.14"), ("13.0", "8.74"))
    assert sample_3 == (("10.0", "7.46"), ("8.0", "6.77"), ("13.0", "12.74"))
    assert sample_4 == (("8.0", "6.58"), ("8.0", "5.76"), ("8.0", "7.71"))

    # Compute statistical summary values -- mean of y and other (not shown) properties.

    # Apply float() function conversion as the last step for mean calculation.
    mean_y_sample_1 = sum(float(pair[1]) for pair in sample_1) / len(sample_1)

    assert mean_y_sample_1 == pytest.approx(7.52333)


.                                                                                            [100%]
1 passed in 0.01s


## Using stateful mappings

* **Treat mappings carefully because they are stateful**
    * Python dictionaries (`dict`) and related mapping types in the `collections` module are stateful collections.
    * From a functional programming perspective, distinguish between:
        * *Frozen dictionaries*: constructed once and subsequently treated as immutable.
        * *Stateful dictionaries*: deliberately modified as the program executes.
    * Python does not provide a particularly convenient built-in immutable mapping for this purpose.
        * The `collections.Mapping` abstract class is immutable, but it's not something we can use trivially.
    * A practical convention for treating a dictionary as immutable is:
        * Assign the dictionary variable only once.
        * Do not subsequently call mutation methods such as `.update()` or `.pop()`.
        * Do not delete dictionary entries using `del`.
* **Two use cases for stateful dictionaries**
    1. *Dictionary built once and never updated*
        * Construct the dictionary from an iterable of `(key, value)` tuples using `dict()`.
        * Exploit dictionary hashing to provide efficient key-based lookups.
        * The construction uses the `process(wrap(iterable))` design pattern:
    2. *Dictionary built incrementally*
        * Build the dictionary progressively as data is processed.
        * Can avoid materialising and sorting an intermediate list.
        * Particularly useful for:
            * *Reductions*: accumulating results, e.g., with `collections.Counter`.
            * *Memoisation*: storing previously computed results to avoid repeating computations.
* **Dictionaries provide efficient lookups**
    * Once constructed, a dictionary can be reused for repeated key-to-value transformations.
    * Python dictionaries use hashing to locate entries efficiently.
    * This makes them particularly useful when many lookups are required against a relatively stable mapping.

**Key insight:** The section isn't advocating avoiding mutable dictionaries altogether. Rather, it advocates being deliberate about their use. Prefer constructing a mapping once and treating it as immutable when possible; use incremental mutation when it serves a specific purpose, such as accumulating results or implementing memoisation.

In [59]:
%%ipytest

Color = namedtuple("Color", ("red", "green", "blue", "name"))


def test_frozen_dictionary() -> None:
    colors = (
        Color(red=239, green=222, blue=205, name="Almond"),
        Color(red=205, green=149, blue=117, name="Antique Brass"),
        Color(red=253, green=217, blue=181, name="Apricot"),
        Color(red=197, green=227, blue=132, name="Yellow Green"),
        Color(red=255, green=174, blue=66, name="Yellow Orange"),
    )

    # Materialise the mapping using the process(wrap(iterable)) design pattern.
    # Wrap: transform each input object into a `(key, value)` tuple.
    # Process: pass the resulting iterable to `dict()` to construct the mapping.
    name_map = dict((color.name, color) for color in colors)

    assert name_map["Almond"] == Color(red=239, green=222, blue=205, name="Almond")
    assert name_map["Apricot"] == Color(red=253, green=217, blue=181, name="Apricot")

.                                                                                            [100%]
1 passed in 0.01s


## Using the bisect module to create a mapping

* Instead of creating a `dict` mapping to achieve fast mapping, we can use the `bisect` module.
    * We have to create a sorted object for searching.
    * To be perfectly compatible with the `dict` mapping, we can use `collections.Mapping` as the base class.
* The `dict` mapping uses a hash to locate items almost immediately, but requires allocating a fairly large block of memory -- the extra capacity required by its hash-table structure.
* The `bisect` mapping does a search, which doesn't require as much memory, but still with good performance ($O(\log{}n)$).
* Another option is to start with the source code for the `collections.OrderedDict` class, change the superclass to `Mapping` instead of `MutableMapping`, and remove all of the methods that implement mutability.

In [61]:
%%ipytest


class StaticMapping(Mapping):
    def __init__(self, iterable: Iterable[tuple[Any, Any]]) -> None:
        # Note: Does *not* handle duplicate keys!
        self._data = tuple(sorted(iterable))
        self._keys = tuple(key for key, _ in self._data)

    def __getitem__(self, key: Any) -> Any:
        ix = bisect.bisect_left(self._keys, key)
        if ix != len(self._keys) and self._keys[ix] == key:
            return self._data[ix][1]

        msg = f"{key!r} not found"
        raise KeyError(msg)

    def __iter__(self) -> Iterator[Any]:
        return iter(self._keys)

    def __len__(self) -> int:
        return len(self._keys)


def test_static_mapping() -> None:
    colors = (
        Color(red=197, green=227, blue=132, name="Yellow Green"),
        Color(red=205, green=149, blue=117, name="Antique Brass"),
        Color(red=253, green=217, blue=181, name="Apricot"),
        Color(red=255, green=174, blue=66, name="Yellow Orange"),
        Color(red=239, green=222, blue=205, name="Almond"),
    )

    name_color = tuple((color.name, color) for color in colors)
    static_map = StaticMapping(name_color)

    assert static_map["Almond"] == Color(red=239, green=222, blue=205, name="Almond")
    assert static_map["Apricot"] == Color(red=253, green=217, blue=181, name="Apricot")


.                                                                                            [100%]


1 passed in 0.01s


## Using stateful sets

* Python provides two relevant set collections:
    * `set`: a mutable collection used to accumulate items.
    * `frozenset`: an immutable collection used to optimise membership searches.
* **Using `frozenset` for efficient membership searches**
    * A `frozenset` is created from an iterable, using `frozenset(some_iterable)`.
    * It provides a very fast `in` operator for checking whether an item belongs to the collection.
    * A typical approach is to gather data, create a `frozenset`, and then use it to process other data.
* **Using mutable sets for accumulating and memoising values**
    * A regular `set` can accumulate items as data is processed.
    * Some algorithms benefit from memoising a set of values: recording values encountered during processing so they can be used again.
    * Whereas memoisation using a mapping is suitable when associating domain values with range values, some algorithms benefit from simply maintaining a growing set of values.

**Key insight:** Use `frozenset` when you have a collection of values that you want to search efficiently, and a mutable `set` when an algorithm needs to accumulate and remember values as it processes data.